# Supplementary Figure 7 | QTL ascertainment and eGene/sGene annotation odds.

**a.** Distributions of median RNA-seq read counts for novel lncRNA, existing lncRNA, and protein-coding genes. Genes with median read count 0 are omitted. Lines are per-tissue distributions, colored by GTEx tissue. Dotted vertical line is median of per-tissue medians (novel lncRNA = 3; existing lncRNA = 11; protein-coding = 8.9*102). Novel lncRNAs are expressed at lower levels than existing lncRNAs and protein-coding genes.

**b.** Distributions of the number of tested intron phenotypes for novel lncRNA, existing lncRNA, and protein-coding genes. Genes with median tested intron count 0 are omitted. Lines are per-tissue distributions, colored by GTEx tissue. Dotted vertical line is median of per-tissue medians (novel lncRNA = 2; existing lncRNA = 3; protein-coding = 7). Fewer splicing phenotypes are tested for lncRNAs, particularly novel lncRNAs, than for protein-coding genes.

**c.** Percentage of genes that are eGenes or sGenes (FDR 5%), by biotype (novel lncRNA, existing lncRNA, protein-coding). Dots are per-tissue single-tissue rates, colored by GTEx tissue. Outline bars are the rate across all tissues, with the edge colored by biotype. Novel lncRNAs are mapped as eGenes and sGenes less often than previously annotated lncRNAs (eGenes: 52% versus 73%, sGenes: 10% versus 26%).

**d.** Unique genes that are e/sGenes (FDR 5%), by biotype and QTL modality. Bars are eGene-only, both eGene and sGene, or sGene-only.

**e.** Gene-level probability a gene is an eGene or sGene in any tissue (FDR 5%) versus maximum median read counts in any tissue, maximum number of tested introns in any tissue, number of tissues eQTL-tested, or number of tissues sQTL-tested, stratified by biotype. Points are means in 20 equal-frequency bins; shaded bands are Wald 95% CIs for the binomial proportion. e/sGene rate increases with expression, intron number, and QTL ascertainment but is largely similar across biotypes at matched expression or tissue number.

**f.** Logistic odds ratios for any-tissue eGene (circles) or sGene (squares) status among annotated lncRNAs. Grey points are one-at-a-time annotation models; dark points are joint models across all annotations and additionally including expression (as log10(TPM + 1) where TPM is the maximum median tissue TPM in any tissue), and number of splicing phenotypes for sGenes (log10 maximum tested introns in any tissue) as covariates. Points, logistic-regression odds ratios (ORs); error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001).


## Setup


In [ ]:
from pathlib import Path
import glob
import gzip
import json
import warnings

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from tqdm import tqdm

plt.rc("font", size=6)
plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 6.2, 300

n_prob_bins = 20
prob_bin_min_n = 25
prob_ci_z = 1.96

gene_type_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]
gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein coding",
}
gene_type_palette = {
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
}

gene_modality_rows = [
    ("protein_coding", "protein-\ncoding"),
    ("novel_lncRNA", "novel\nlncRNA"),
    ("existing_lncRNA", "existing\nlncRNA"),
]
gene_modality_order = ["egene", "both", "sgene"]
gene_modality_label = {"egene": "eGene", "both": "both", "sgene": "sGene"}
_eq_rgb = np.array([0x3B, 0x7E, 0xA1]) / 255.0
_sq_rgb = np.array([0xE8, 0xA8, 0x38]) / 255.0
_both_rgb = 0.5 * (_eq_rgb + _sq_rgb)
gene_modality_color = {
    "egene": "#3B7EA1",
    "both": "#%02x%02x%02x" % tuple(int(round(255 * x)) for x in _both_rgb),
    "sgene": "#E8A838",
}


## Paths

In [ ]:
# Portal QTL / expression inputs.
eqtl_dir = Path("path/to/v11_eqtl_susie")
sqtl_dir = Path("path/to/v11_sqtl_susie")
sqtl_pheno_dir = Path("path/to/v11_sqtl_phenotype_matrices")
sample_meta_file = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
qtl_tissue_ids_file = Path("path/to/qtl_tissue_ids.txt")
gtex_colors_file = Path("refs/colors.json")
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")
lncrna_cat_file = Path("path/to/lncrna_category.txt")
median_tpm_file = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
median_counts_file = Path("path/to/tissue_median_counts.txt")
egene_qval_threshold = 0.05
sgene_qval_threshold = 0.05
fig_dir = Path("figures")


## Load data

### QTL tissue ids

In [ ]:
qtl_tissue_ids = set(pd.read_table(qtl_tissue_ids_file, header=None)[0])

### Tissue colors

In [ ]:
with open(gtex_colors_file) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
tissue_df["tissue_color_hex"] = "#" + tissue_df["tissue_color_hex"]
tissue_df = tissue_df[tissue_df["tissue_file_name"].isin(qtl_tissue_ids)].copy()
tissue_color = dict(zip(tissue_df["tissue_file_name"], tissue_df["tissue_color_hex"]))
print(f"QTL tissues: {len(qtl_tissue_ids)}; with colors: {len(tissue_color)}")


### RNA-seq sample sizes


In [ ]:
gtex_sample_meta = pd.read_csv(sample_meta_file, sep="\t", low_memory=False)
rnaseq_samples = gtex_sample_meta[gtex_sample_meta["SMAFRZE"] == "RNASEQ"][["SAMPID", "SMTSD"]].copy()
rnaseq_samples["tissue_file_name"] = (
    rnaseq_samples["SMTSD"]
    .str.replace("(", "", regex=False).str.replace(")", "", regex=False)
    .str.replace(" - ", "_").str.replace(" ", "_")
)
tissue_sample_size = (
    rnaseq_samples[rnaseq_samples["tissue_file_name"].isin(qtl_tissue_ids)]
    .groupby("tissue_file_name")["SAMPID"].nunique()
    .rename("n_samples")
    .reset_index()
)
print(f"RNASEQ sample sizes for {len(tissue_sample_size)} QTL tissues")

### GENCODE classification


In [ ]:
gencode_genes = pd.read_table(gencode_class_file, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"].copy()
print(f"GENCODE genes (excl. artifact): {len(gencode_genes):,}")


### lncRNA category labels


In [ ]:
lnc_cat_all = pd.read_table(
    lncrna_cat_file,
    usecols=[
        "gene_id", "gene_id_stable", "is_mirna_host", "is_ncORF",
        "expressed_dgtex", "expressed_macaque", "expressed_marmoset",
        "is_tissue_specific", "peak_tissue",
        "macaque_tissue_concordant", "marmoset_tissue_concordant",
        "primate_cons_exon_unique", "primate_cons_sj_unique",
        "tss_e2g_enh",
    ],
).drop_duplicates("gene_id_stable")
lncrna_ids = set(
    gencode_genes.loc[
        gencode_genes["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"]),
        "gene_id_stable",
    ]
)
lnc_cat_all = lnc_cat_all[lnc_cat_all["gene_id_stable"].isin(lncrna_ids)].copy()
print(f"lncRNA category rows: {len(lnc_cat_all):,}")


### Portal gene median TPM (panel f covariates; QTL tissues only)

In [ ]:
with gzip.open(median_tpm_file, "rt") as fh:
    fh.readline()
    fh.readline()
    raw = pd.read_csv(fh, sep="\t")
median_tpm = raw.set_index("Name").drop(columns=["Description"], errors="ignore")

# Portal GCT columns are tissue file stems (same encoding as qtl_tissue_ids).
def _portal_col_to_stem(col):
    if col in qtl_tissue_ids:
        return col
    alt = col.replace("-", "_")
    if alt in qtl_tissue_ids:
        return alt
    return None

stem_cols = {}
for c in median_tpm.columns:
    stem = _portal_col_to_stem(c)
    if stem is not None:
        stem_cols[c] = stem
median_tpm = median_tpm.rename(columns=stem_cols)
median_tpm = median_tpm[[c for c in median_tpm.columns if c in qtl_tissue_ids]].copy()

_missing = sorted(qtl_tissue_ids - set(median_tpm.columns))
if _missing:
    raise ValueError(
        f"Portal median TPM missing {len(_missing)}/{len(qtl_tissue_ids)} QTL tissues: {_missing}"
    )

max_tpm_by_v47 = median_tpm.max(axis=1)
print(
    f"Portal median TPM (QTL tissues): {median_tpm.shape[0]:,} genes × "
    f"{median_tpm.shape[1]} tissues (from {median_tpm_file.name})"
)


### Tissue median read counts

In [ ]:
tissue_medians = pd.read_table(median_counts_file, index_col=0)
qtl_cols = [c for c in tissue_medians.columns if c in qtl_tissue_ids]
qtl_counts = tissue_medians[qtl_cols]
print(f"median read counts: {qtl_counts.shape[0]:,} genes × {len(qtl_cols)} QTL tissues")

### Compute — eGene / sGene (gene-level + gene–tissue)


In [ ]:
# One pass per modality builds gene-level and gene–tissue tables.
def _build_egene_sgene_tables(qtl_dir, file_suffix, qval_threshold):
    files = sorted(qtl_dir.glob(f"*.{file_suffix}.txt.gz"))
    if not files:
        raise FileNotFoundError(f"No *.{file_suffix}.txt.gz under {qtl_dir}")
    print(f"Found {len(files)} files in {qtl_dir}")

    long_records = []
    summary_records = []
    for fpath in tqdm(files, desc=file_suffix):
        tissue = fpath.name.split(".v11.")[0]
        if tissue not in qtl_tissue_ids:
            continue
        n_samp = tissue_sample_size.loc[
            tissue_sample_size["tissue_file_name"] == tissue, "n_samples"
        ].values
        n_samp = int(n_samp[0]) if len(n_samp) else np.nan

        df = pd.read_csv(fpath, sep="\t", usecols=["gene_id", "qval", "slope"])
        df = (
            df.sort_values("qval")
            .groupby("gene_id", as_index=False)
            .first()
        )
        df["tissue"] = tissue
        df["n_samples"] = n_samp
        df["gene_id_stable"] = df["gene_id"].str.split(".").str[0]
        df["is_qtl"] = df["qval"] < qval_threshold

        long_records.append(df[["gene_id_stable", "tissue", "is_qtl"]])
        summary_records.append(df[["gene_id", "qval", "slope", "tissue", "n_samples"]])

    if not summary_records:
        raise ValueError(f"No QTL tissues found under {qtl_dir} for {file_suffix}")

    long_df = pd.concat(long_records, ignore_index=True)
    sum_long = pd.concat(summary_records, ignore_index=True)
    sum_long["is_qtl"] = sum_long["qval"] < qval_threshold

    best = (
        sum_long.sort_values("qval")
        .groupby("gene_id", as_index=False)
        .first()
        .rename(columns={
            "qval": "best_qval", "slope": "best_slope",
            "tissue": "best_tissue", "n_samples": "max_sample_size",
        })
    )
    n_tiss = (
        sum_long.loc[sum_long["is_qtl"]]
        .groupby("gene_id")["tissue"].nunique()
        .reset_index().rename(columns={"tissue": "n_tissues_qtl"})
    )
    max_n = (
        sum_long.groupby("gene_id")["n_samples"].max()
        .reset_index().rename(columns={"n_samples": "max_sample_size_tested"})
    )
    summary = best.merge(n_tiss, on="gene_id", how="left")
    summary = summary.merge(max_n, on="gene_id", how="left")
    summary["n_tissues_qtl"] = summary["n_tissues_qtl"].fillna(0).astype(int)
    summary["is_qtl_any"] = summary["best_qval"] < qval_threshold
    return summary, long_df


egene_summary, egene_long = _build_egene_sgene_tables(
    eqtl_dir, "v11.eGenes", egene_qval_threshold
)
sgene_summary, sgene_long = _build_egene_sgene_tables(
    sqtl_dir, "v11.sGenes", sgene_qval_threshold
)


### eGene gene-level summary

In [ ]:
print(f"eGene gene-level summary: {egene_summary.shape}")


### sGene gene-level summary

In [ ]:
print(f"sGene gene-level summary: {sgene_summary.shape}")


### Compute — tested intron counts per gene–tissue


In [ ]:
junction_rows = []
pheno_files = sorted(glob.glob(str(sqtl_pheno_dir / "*.v11.sQTL_phenotype_groups.txt.gz")))
if not pheno_files:
    raise FileNotFoundError(f"No *.v11.sQTL_phenotype_groups.txt.gz under {sqtl_pheno_dir}")
for fpath in tqdm(pheno_files, desc="sQTL phenotype groups"):
    tissue = Path(fpath).name.split(".v11.")[0]
    if tissue not in qtl_tissue_ids:
        continue
    pg = pd.read_csv(fpath, sep="\t", header=None, names=["phenotype_id", "gene_id"])
    counts = pg.groupby("gene_id").size().rename("n_junctions").reset_index()
    counts["tissue"] = tissue
    junction_rows.append(counts)
if not junction_rows:
    raise ValueError(f"No QTL tissues found under {sqtl_pheno_dir}")
n_junctions = pd.concat(junction_rows, ignore_index=True)
n_junctions["gene_id_stable"] = n_junctions["gene_id"].str.split(".").str[0]


### Tested intron counts per gene and tissue

In [ ]:
print(
    f"tested introns: {n_junctions.shape}; "
    f"median={n_junctions['n_junctions'].median():.0f}, max={n_junctions['n_junctions'].max()}"
)


### eGene calls per gene and tissue

In [ ]:
print(f"eGene gene–tissue rows: {len(egene_long):,}")


### sGene calls per gene and tissue

In [ ]:
print(f"sGene gene–tissue rows: {len(sgene_long):,}")


## Run analysis


### Master gene metadata


In [ ]:
max_median_counts = qtl_counts.max(axis=1).rename("max_median_counts")
gene_expr_stats = max_median_counts.rename_axis("gene_id").reset_index()
gene_expr_stats["gene_id_stable"] = gene_expr_stats["gene_id"].str.split(".").str[0]

gene_meta = gencode_genes[[
    "gene_id_stable", "gene_id_v47", "gene_type_grouped",
    "in_v47_full",
]].copy()
gene_meta = gene_meta[gene_meta["in_v47_full"]].copy()
gene_meta = gene_meta[gene_meta["gene_type_grouped"].isin(["protein_coding", "novel_lncRNA", "existing_lncRNA"])].copy()

gene_meta = gene_meta.merge(
    gene_expr_stats[["gene_id_stable", "max_median_counts"]],
    on="gene_id_stable", how="left",
)

eg = egene_summary.copy()
eg["gene_id_stable"] = eg["gene_id"].str.split(".").str[0]
gene_meta = gene_meta.merge(
    eg[["gene_id_stable", "is_qtl_any"]].rename(columns={"is_qtl_any": "is_egene"}),
    on="gene_id_stable", how="left",
)
gene_meta["is_egene"] = gene_meta["is_egene"].fillna(False).astype(bool)

sg = sgene_summary.copy()
sg["gene_id_stable"] = sg["gene_id"].str.split(".").str[0]
gene_meta = gene_meta.merge(
    sg[["gene_id_stable", "is_qtl_any"]].rename(columns={"is_qtl_any": "is_sgene"}),
    on="gene_id_stable", how="left",
)
gene_meta["is_sgene"] = gene_meta["is_sgene"].fillna(False).astype(bool)

print(f"Master dataframe (v47 PC+lncRNA): {gene_meta.shape}")

### Gene-level analysis tables


In [ ]:
junc_gene = (
    n_junctions.groupby("gene_id_stable", as_index=False)["n_junctions"]
    .max()
    .rename(columns={"n_junctions": "max_n_junctions"})
)

n_eqtl_tested = (
    egene_long.groupby("gene_id_stable")["tissue"].nunique()
    .rename("n_tissues_tested_eqtl")
)
n_sqtl_tested = (
    sgene_long.groupby("gene_id_stable")["tissue"].nunique()
    .rename("n_tissues_tested_sqtl")
)

genes = gene_meta.drop_duplicates("gene_id_stable").copy()
genes = genes.merge(junc_gene, on="gene_id_stable", how="left")
genes["max_n_junctions"] = genes["max_n_junctions"].fillna(0)
genes["n_tissues_tested_eqtl"] = genes["gene_id_stable"].map(n_eqtl_tested).fillna(0).astype(int)
genes["n_tissues_tested_sqtl"] = genes["gene_id_stable"].map(n_sqtl_tested).fillna(0).astype(int)
genes["is_egene"] = genes["is_egene"].astype(int)
genes["is_sgene"] = genes["is_sgene"].astype(int)

model1_genes = genes[genes["gene_type_grouped"].isin(gene_type_order)].copy()
print(f"Panel c–e genes (biotype): {len(model1_genes):,}")


### Per-tissue eGene/sGene rates and eGene/both/sGene counts (panels c–d)


In [ ]:
def build_qtl_pct_summary(long_df, any_col):
    rows, hollows = [], []
    genes_bt = model1_genes[["gene_id_stable", "gene_type_grouped", any_col]].copy()
    sig = (
        long_df.loc[long_df["is_qtl"], ["gene_id_stable", "tissue"]]
        .drop_duplicates()
    )
    tissue_gene_sets = {
        tissue: set(gids)
        for tissue, gids in sig.groupby("tissue")["gene_id_stable"]
    }
    for idx, gt in enumerate(gene_type_order):
        sel = genes_bt["gene_type_grouped"].eq(gt)
        n = int(sel.sum())
        gids = set(genes_bt.loc[sel, "gene_id_stable"])
        for tissue, gset in tissue_gene_sets.items():
            rows.append((gt, tissue, 100 * len(gset & gids) / n if n else 0.0))
        hollows.append((gt, 100 * genes_bt.loc[sel, any_col].sum() / n if n else 0.0, idx))
    return pd.DataFrame(rows, columns=["gene_type_grouped", "tissue", "pct"]), hollows


egene_pct_summary, egene_hollows = build_qtl_pct_summary(egene_long, "is_egene")
sgene_pct_summary, sgene_hollows = build_qtl_pct_summary(sgene_long, "is_sgene")

gene_modality_stacks = {}
for key, lab in gene_modality_rows:
    sub = model1_genes[model1_genes["gene_type_grouped"] == key]
    eg_only = int(((sub["is_egene"] == 1) & (sub["is_sgene"] == 0)).sum())
    both = int(((sub["is_egene"] == 1) & (sub["is_sgene"] == 1)).sum())
    sg_only = int(((sub["is_egene"] == 0) & (sub["is_sgene"] == 1)).sum())
    gene_modality_stacks[key] = {
        "egene": eg_only,
        "both": both,
        "sgene": sg_only,
        "total": eg_only + both + sg_only,
    }


### Expression and tested-intron plot tables


In [ ]:
genes_bt = (
    model1_genes[["gene_id_stable", "gene_type_grouped"]]
    .drop_duplicates("gene_id_stable")
    .rename(columns={"gene_type_grouped": "biotype"})
)

expr_long = (
    qtl_counts.reset_index()
    .rename(columns={"Name": "gene_id"})
    .melt(id_vars=["gene_id"], var_name="tissue", value_name="Expression")
)
expr_long["gene_id_stable"] = expr_long["gene_id"].str.split(".").str[0]
expr_plot_df = expr_long.merge(genes_bt, on="gene_id_stable", how="inner")
expr_plot_df = expr_plot_df[expr_plot_df["Expression"] > 0].copy()

tissues_junc = sorted(set(n_junctions["tissue"]) & set(qtl_tissue_ids))
gene_tissue = (
    genes_bt.assign(_k=1)
    .merge(pd.DataFrame({"tissue": tissues_junc, "_k": 1}), on="_k")
    .drop(columns="_k")
)
junc_counts = (
    n_junctions.merge(genes_bt[["gene_id_stable"]], on="gene_id_stable", how="inner")
    .groupby(["gene_id_stable", "tissue"], as_index=False)["n_junctions"]
    .sum()
)
junc_tissue = gene_tissue.merge(junc_counts, on=["gene_id_stable", "tissue"], how="left")
junc_tissue["n_junctions"] = junc_tissue["n_junctions"].fillna(0).astype(int)
junc_plot_df = junc_tissue[junc_tissue["n_junctions"] > 0].copy()

print(f"Expression plot rows (counts>0): {len(expr_plot_df):,}")
print(f"Tested-intron plot rows (>0): {len(junc_plot_df):,}")

### Binned eGene/sGene probabilities (panel e)


In [ ]:
def binned_prob(df, xcol, ycol, n_bins=n_prob_bins, min_n=prob_bin_min_n, z=prob_ci_z):
    d = df[[xcol, ycol]].dropna().copy()
    d = d[np.isfinite(d[xcol])]
    if d.empty:
        return pd.DataFrame(columns=["x", "p", "n", "p_lo", "p_hi"])
    d["_rank"] = d[xcol].rank(method="first")
    d["_bin"] = pd.qcut(d["_rank"], q=n_bins, duplicates="drop")
    out = (
        d.groupby("_bin", observed=True)
        .agg(x=(xcol, "median"), p=(ycol, "mean"), n=(ycol, "size"))
        .reset_index(drop=True)
    )
    out = out[out["n"] >= min_n].copy()
    se = np.sqrt(out["p"] * (1.0 - out["p"]) / out["n"])
    out["p_lo"] = np.clip(out["p"] - z * se, 0.0, 1.0)
    out["p_hi"] = np.clip(out["p"] + z * se, 0.0, 1.0)
    return out

prob_specs = [
    ("max_median_counts", "is_egene", "P(eGene)", "Max tissue median reads"),
    ("max_median_counts", "is_sgene", "P(sGene)", "Max tissue median reads"),
    ("max_n_junctions", "is_sgene", "P(sGene)", "Max tested introns"),
    ("n_tissues_tested_eqtl", "is_egene", "P(eGene)", "N tissues tested (eQTL)"),
    ("n_tissues_tested_sqtl", "is_sgene", "P(sGene)", "N tissues tested (sQTL)"),
]
prob_curves = {}
for gt in tqdm(gene_type_order, desc="binned e/sGene probs"):
    gt_df = model1_genes[model1_genes["gene_type_grouped"] == gt]
    for xcol, ycol, _, _ in prob_specs:
        key = (gt, xcol, ycol)
        prob_curves[key] = binned_prob(gt_df, xcol, ycol)
        print(
            f"{gt} {ycol}~{xcol}: {len(prob_curves[key])} bins, "
            f"n={int(prob_curves[key]['n'].sum()) if len(prob_curves[key]) else 0}"
        )

### eGene/sGene annotation ORs (panel f)

Same categories and individual vs ascertainment-adjusted individual models as Figure 4d. Outcomes are any-tissue eGene / sGene among annotated lncRNAs. Adjusted eGene covars: log10 max TPM + N eQTL tissues; adjusted sGene additionally includes log10 max tested introns.


In [ ]:
# Figure 4d categories; outcomes = any-tissue eGene / sGene among annotated lncRNAs
lnc_by_stable = lnc_cat_all.set_index("gene_id_stable")
stable_to_v47 = (
    gencode_genes.drop_duplicates("gene_id_stable").set_index("gene_id_stable")["gene_id_v47"]
)

nontestis_ok = ~lnc_by_stable["peak_tissue"].eq("Testis")
# expressed_macaque/marmoset already require an ortholog id (classify_lncrna)
primate_expr_nontestis = (
    lnc_by_stable["expressed_macaque"].astype(bool)
    | lnc_by_stable["expressed_marmoset"].astype(bool)
) & nontestis_ok
dgtex = lnc_by_stable["expressed_dgtex"].astype(bool)
mac_tc = lnc_by_stable["macaque_tissue_concordant"].astype(bool)
mar_tc = lnc_by_stable["marmoset_tissue_concordant"].astype(bool)
ts = lnc_by_stable["is_tissue_specific"].astype(bool)
testis_peak = lnc_by_stable["peak_tissue"].eq("Testis")
ts_testis = ts & testis_peak
ts_nontestis = ts & nontestis_ok
nhp_match = mac_tc | mar_tc
tc_testis = ts_testis & nhp_match
tc_nontestis = ts_nontestis & nhp_match

primate_cons_exon_unique = lnc_by_stable["primate_cons_exon_unique"].astype(bool)
primate_cons_sj_unique = lnc_by_stable["primate_cons_sj_unique"].astype(bool)
tss_e2g_enh = lnc_by_stable["tss_e2g_enh"].astype(bool)

egene_ids = set(genes.loc[genes["is_egene"].astype(bool), "gene_id_stable"])
sgene_ids = set(genes.loc[genes["is_sgene"].astype(bool), "gene_id_stable"])
n_eqtl_tested_by_stable = egene_long.groupby("gene_id_stable")["tissue"].nunique()
n_sqtl_tested_by_stable = sgene_long.groupby("gene_id_stable")["tissue"].nunique()
max_n_junc_by_stable = n_junctions.groupby("gene_id_stable")["n_junctions"].max()

or_traits = [
    ("tss_e2g_enh", "enhancer", None),
    ("is_mirna_host", "miRNA\nhost", None),
    ("is_ncORF", "translated\nncORF", None),
    ("primate_cons_exon_unique", "primate\nexon\ncons.", None),
    ("primate_cons_sj_unique", "primate\nsplice junct.\ncons.", None),
    ("primate_expr", "primate\nexpr.", None),
    ("dev_expr", "dev.\nexpr.", None),
    ("ts_testis", "TS\n(testis)", "ts_vs_not_ts_testis"),
    ("ts_nontestis_only", "TS\n(non-testis)", "ts_vs_not_ts_nontestis"),
    ("tc_testis", "TC\n(testis)", "ts_vs_not_tc_testis"),
    ("tc_nontestis", "TC\n(non-testis)", "ts_vs_not_tc_nontestis"),
]

annot_group = "annotation"
joint_e_group = "annotation + expr + N tissues"
joint_s_group = "annotation + expr + N tissues + N introns"
joint_covars_e = "log_max_tpm + n_tissues_tested_eqtl"
joint_covars_s = "log_max_tpm + n_tissues_tested_sqtl + log_max_n_junctions"


def build_outcome_gld(outcome_ids, outcome_col):
    gld = lnc_cat_all.copy()
    gld[outcome_col] = gld["gene_id_stable"].isin(outcome_ids).astype(int)
    gene_v47 = gld["gene_id_stable"].map(stable_to_v47)
    gld["log_max_tpm"] = np.log10(gene_v47.map(max_tpm_by_v47).fillna(0) + 1)
    gld["n_tissues_tested_eqtl"] = (
        gld["gene_id_stable"].map(n_eqtl_tested_by_stable).fillna(0).astype(float)
    )
    gld["n_tissues_tested_sqtl"] = (
        gld["gene_id_stable"].map(n_sqtl_tested_by_stable).fillna(0).astype(float)
    )
    gld["log_max_n_junctions"] = np.log10(
        gld["gene_id_stable"].map(max_n_junc_by_stable).fillna(0).astype(float) + 1
    )
    gld["tss_e2g_enh"] = gld["gene_id_stable"].map(tss_e2g_enh).fillna(False).astype(int)
    gld["is_mirna_host"] = gld["is_mirna_host"].astype(int)
    gld["is_ncORF"] = gld["is_ncORF"].astype(int)
    gld["primate_cons_exon_unique"] = (
        gld["gene_id_stable"].map(primate_cons_exon_unique).fillna(False).astype(int)
    )
    gld["primate_cons_sj_unique"] = (
        gld["gene_id_stable"].map(primate_cons_sj_unique).fillna(False).astype(int)
    )
    gld["primate_expr"] = gld["gene_id_stable"].map(primate_expr_nontestis).fillna(False).astype(int)
    gld["dev_expr"] = gld["gene_id_stable"].map(dgtex).fillna(False).astype(int)
    _ts_testis = gld["gene_id_stable"].map(ts_testis).fillna(False).astype(int)
    _ts_nontestis = gld["gene_id_stable"].map(ts_nontestis).fillna(False).astype(int)
    gld["tc_testis"] = gld["gene_id_stable"].map(tc_testis).fillna(False).astype(int)
    gld["tc_nontestis"] = gld["gene_id_stable"].map(tc_nontestis).fillna(False).astype(int)
    gld["ts_testis"] = ((_ts_testis == 1) & (gld["tc_testis"] == 0)).astype(int)
    gld["ts_nontestis_only"] = ((_ts_nontestis == 1) & (gld["tc_nontestis"] == 0)).astype(int)
    gld["ts_vs_not_ts_testis"] = (
        (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_ts_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["tc_testis"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_testis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_nontestis"] == 0)
    ).astype(int)
    gld["ts_vs_not_tc_nontestis"] = (
        (gld["ts_testis"] == 0) & (gld["ts_nontestis_only"] == 0) & (gld["tc_testis"] == 0)
    ).astype(int)
    return gld


def fit_trait_or(work, outcome_col, trait_col, subset_col=None, covars=""):
    d = work[work[subset_col] == 1].copy() if subset_col is not None else work.copy()
    if d.empty or d[outcome_col].nunique() < 2 or d[trait_col].nunique() < 2:
        return None
    n_pos = int(d[outcome_col].sum())
    if n_pos < 3 or len(d) - n_pos < 3:
        return None
    is_binary = set(pd.unique(d[trait_col].dropna())).issubset({0, 1, 0.0, 1.0, True, False})
    if is_binary:
        pos = d[outcome_col] == 1
        n_pos_trait = int(d.loc[pos, trait_col].sum())
        if d[trait_col].sum() < 3 or n_pos_trait == 0 or (n_pos - n_pos_trait) == 0:
            return None
    covar_cols = [c.strip() for c in covars.split("+")] if covars else []
    for c in covar_cols:
        if c not in d.columns or d[c].nunique() < 2:
            return None
    formula = f"{outcome_col} ~ {trait_col}" + (f" + {covars}" if covars else "")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.logit(formula, data=d).fit(disp=False, maxiter=200)
    except Exception:
        return None
    if trait_col not in fit.params.index:
        return None
    ci = fit.conf_int().loc[trait_col]
    or_hat = float(np.exp(fit.params[trait_col]))
    ci_low, ci_high = float(np.exp(ci.iloc[0])), float(np.exp(ci.iloc[1]))
    if not all(np.isfinite(v) for v in (or_hat, ci_low, ci_high)):
        return None
    return {
        "trait": trait_col, "OR": or_hat, "CI_low": ci_low, "CI_high": ci_high,
        "pvalue": float(fit.pvalues[trait_col]), "n_pos": n_pos,
    }


def compute_individual_ors(gld, outcome_col, traits, group, covars=""):
    rows = []
    for trait_col, trait_label, subset_col in traits:
        row = fit_trait_or(
            gld, outcome_col, trait_col, subset_col=subset_col, covars=covars
        ) or {
            "trait": trait_col, "OR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
            "pvalue": np.nan,
            "n_pos": int(gld[outcome_col].sum()) if subset_col is None else int(
                gld.loc[gld[subset_col] == 1, outcome_col].sum()
            ),
        }
        row["label"] = trait_label
        row["group"] = group
        rows.append(row)
    return pd.DataFrame(rows)


gld_e = build_outcome_gld(egene_ids, "has_qtl")
gld_s = build_outcome_gld(sgene_ids, "has_qtl")

indiv_e = compute_individual_ors(gld_e, "has_qtl", or_traits, annot_group)
indiv_e["modality"] = "eGene"
indiv_s = compute_individual_ors(gld_s, "has_qtl", or_traits, annot_group)
indiv_s["modality"] = "sGene"

adj_e = compute_individual_ors(
    gld_e, "has_qtl", or_traits, joint_e_group, covars=joint_covars_e
)
adj_e["modality"] = "eGene"
adj_s = compute_individual_ors(
    gld_s, "has_qtl", or_traits, joint_s_group, covars=joint_covars_s
)
adj_s["modality"] = "sGene"

or_df = pd.concat([indiv_e, indiv_s, adj_e, adj_s], ignore_index=True)

indiv_e_or = (
    or_df.loc[(or_df["group"] == annot_group) & (or_df["modality"] == "eGene")]
    .drop_duplicates("label")
    .set_index("label")["OR"]
)
panel_d_cat_order = sorted(
    [lab for t, lab, _ in or_traits],
    key=lambda lab: (
        0, -float(indiv_e_or[lab])
    ) if lab in indiv_e_or.index and np.isfinite(indiv_e_or[lab]) else (1, 0.0),
)
print(
    f"eGene: n={len(gld_e):,}, n_pos={int(gld_e['has_qtl'].sum()):,}, "
    f"adj covars={joint_covars_e}"
)
print(
    f"sGene: n={len(gld_s):,}, n_pos={int(gld_s['has_qtl'].sum()):,}, "
    f"adj covars={joint_covars_s}"
)


## Make figure


In [ ]:
biotypes = [b for b in gene_type_order if b in set(junc_plot_df["biotype"])]
n_bt = len(biotypes)

expr_pos = expr_plot_df.loc[expr_plot_df["Expression"] > 0, "Expression"].to_numpy(dtype=float)
expr_bins = np.logspace(np.log10(0.5), np.log10(float(np.nanmax(expr_pos))), 60)
sj_bins = np.arange(0.5, int(junc_plot_df["n_junctions"].max()) + 1.5, 1)

# Nest legends inside e/f rows so they don't collide with the row above
fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
outer = fig.add_gridspec(
    5, 1,
    height_ratios=[0.9, 0.9, 0.7, 1.15, 1.35],
    hspace=0.5,
    left=0.08, right=0.98, top=0.96, bottom=0.07,
)

tick_fs = 4.5
label_fs = 5.0
title_fs = 5.0
# Match a/b/c column widths and gaps so the three biotype columns align
col_ratios = [1.0, 1.0, 1.15]
col_wspace = 0.45

# Panel a — tissue median read-count distributions
gs_a = outer[0].subgridspec(1, n_bt, width_ratios=col_ratios, wspace=col_wspace)
axes_a = [fig.add_subplot(gs_a[0, j]) for j in range(n_bt)]
for j, bt in enumerate(biotypes):
    ax = axes_a[j]
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(expr_bins[0], expr_bins[-1])
    tissue_medians_e = []
    for tissue, tdf in expr_plot_df.loc[expr_plot_df["biotype"] == bt].groupby("tissue", sort=False):
        vals = tdf["Expression"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals) < 5:
            continue
        tissue_medians_e.append(np.median(vals))
        counts, edges = np.histogram(vals, bins=expr_bins)
        ax.stairs(counts, edges, color=tissue_color.get(tissue, "0.5"), lw=0.7, alpha=0.75)
    med_e = float(np.median(tissue_medians_e)) if tissue_medians_e else np.nan
    if np.isfinite(med_e):
        ax.axvline(med_e, color="k", ls=":", lw=0.6, alpha=0.6)
        ax.annotate(
            f"{med_e:.2g}", xy=(med_e, 0.98), xycoords=("data", "axes fraction"),
            xytext=(3, 0), textcoords="offset points",
            va="top", ha="left", fontsize=3.8, color="0.2",
        )
    ax.set_title(gene_type_pretty[bt], fontsize=title_fs, pad=1.5)
    ax.set_ylabel("genes" if j == 0 else "", fontsize=label_fs)
    ax.set_xlabel("Tissue median read counts", fontsize=label_fs)
    ax.tick_params(axis="both", labelsize=tick_fs, pad=1, length=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panel b — tissue tested-intron distributions
gs_b = outer[1].subgridspec(1, n_bt, width_ratios=col_ratios, wspace=col_wspace)
axes_b = [fig.add_subplot(gs_b[0, j]) for j in range(n_bt)]
for j, bt in enumerate(biotypes):
    ax = axes_b[j]
    ax.set_yscale("log")
    ax.set_xlim(sj_bins[0], sj_bins[-1])
    tissue_medians_j = []
    for tissue, tdf in junc_plot_df.loc[junc_plot_df["biotype"] == bt].groupby("tissue", sort=False):
        vals = tdf["n_junctions"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals) < 5:
            continue
        tissue_medians_j.append(np.median(vals))
        counts, edges = np.histogram(vals, bins=sj_bins)
        ax.stairs(counts, edges, color=tissue_color.get(tissue, "0.5"), lw=0.7, alpha=0.75)
    med_j = float(np.median(tissue_medians_j)) if tissue_medians_j else np.nan
    if np.isfinite(med_j):
        ax.axvline(med_j, color="k", ls=":", lw=0.6, alpha=0.6)
        ax.annotate(
            f"{med_j:.0f}", xy=(med_j, 0.98), xycoords=("data", "axes fraction"),
            xytext=(3, 0), textcoords="offset points",
            va="top", ha="left", fontsize=3.8, color="0.2",
        )
    ax.set_title(gene_type_pretty[bt], fontsize=title_fs, pad=1.5)
    ax.set_ylabel("genes" if j == 0 else "", fontsize=label_fs)
    ax.set_xlabel("Tissue tested introns", fontsize=label_fs)
    ax.tick_params(axis="both", labelsize=tick_fs, pad=1, length=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panels c–d — per-tissue % eGene / sGene + eGene/both/sGene counts
gs_c = outer[2].subgridspec(1, 3, width_ratios=col_ratios, wspace=col_wspace)
axes_c = [fig.add_subplot(gs_c[0, j]) for j in range(2)]
ax_mod = fig.add_subplot(gs_c[0, 2])
for ax, summary, hollows, xlabel in zip(
    axes_c,
    [egene_pct_summary, sgene_pct_summary],
    [egene_hollows, sgene_hollows],
    ["% eGenes (FDR 5%)", "% sGenes (FDR 5%)"],
):
    for i in range(len(gene_type_order)):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color="#fafafa", zorder=0)
    for gt, pct_any, y in hollows:
        ax.barh(
            y, pct_any, left=0, height=0.8,
            edgecolor=gene_type_palette[gt], facecolor=(0, 0, 0, 0),
            linewidth=0.8, zorder=1,
        )
    sns.stripplot(
        data=summary,
        y="gene_type_grouped",
        x="pct",
        order=gene_type_order,
        hue="tissue",
        palette=tissue_color,
        dodge=False,
        size=1.5,
        linewidth=0,
        alpha=0.9,
        zorder=2,
        ax=ax,
        orient="h",
        legend=False,
        jitter=0.2,
    )
    ax.set_yticks(range(len(gene_type_order)))
    ax.set_yticklabels([gene_type_pretty[g] for g in gene_type_order])
    ax.set_ylabel("")
    ax.set_xlabel(xlabel, fontsize=label_fs)
    ax.set_xlim(-1, 100)
    ax.tick_params(axis="y", left=False, labelleft=True, labelsize=tick_fs)
    ax.tick_params(axis="x", labelsize=tick_fs, pad=1, length=2)
    sns.despine(ax=ax, left=True, bottom=False)

edge_lw = 0.7
n_bio = len(gene_modality_rows)
n_mod = len(gene_modality_order)
gene_bar_w = 0.18
gene_offsets = (np.arange(n_mod) - (n_mod - 1) / 2) * gene_bar_w
gene_ymax = 1
for i, (key, _lab) in enumerate(gene_modality_rows):
    s = gene_modality_stacks[key]
    for mod, dx in zip(gene_modality_order, gene_offsets):
        h = s[mod]
        if h <= 0:
            continue
        ax_mod.bar(
            i + dx, h, width=gene_bar_w * 0.9,
            color=gene_modality_color[mod], edgecolor="black", linewidth=edge_lw,
        )
        ax_mod.annotate(
            f"{h:,}",
            xy=(i + dx, h),
            xytext=(0, 2),
            textcoords="offset points",
            va="bottom", ha="center", fontsize=3.5, rotation=90,
        )
        gene_ymax = max(gene_ymax, h)
ax_mod.set_xticks(range(n_bio))
ax_mod.set_xticklabels([lab for _, lab in gene_modality_rows], fontsize=tick_fs)
ax_mod.set_xlabel("")
ax_mod.set_ylabel("Genes", fontsize=label_fs)
ax_mod.set_yscale("log")
ax_mod.set_ylim(10, gene_ymax * 6.0)
ax_mod.set_yticks([10, 100, 1000, 10000])
ax_mod.set_yticklabels(["10", "100", "1k", "10k"])
ax_mod.set_xlim(-0.55, n_bio - 0.45)
ax_mod.tick_params(axis="x", length=0, pad=2)
ax_mod.tick_params(axis="y", labelsize=tick_fs, pad=1, length=2)
ax_mod.spines["top"].set_visible(False)
ax_mod.spines["right"].set_visible(False)
ax_mod.legend(
    handles=[
        Patch(facecolor=gene_modality_color[m], edgecolor="black", linewidth=edge_lw,
              label=gene_modality_label[m])
        for m in gene_modality_order
    ],
    loc="lower center", bbox_to_anchor=(0.5, 1.05),
    frameon=False, ncol=3, handletextpad=0.4, columnspacing=0.9, borderaxespad=0.0,
    fontsize=3.5,
)

# Panel e — legend strip + binned P(eGene)/P(sGene)
gs_d_wrap = outer[3].subgridspec(2, 1, height_ratios=[0.16, 1.0], hspace=0.05)
ax_d_leg = fig.add_subplot(gs_d_wrap[0])
ax_d_leg.set_axis_off()
ax_d_leg.legend(
    handles=[
        plt.Line2D(
            [0], [0], color=gene_type_palette[gt], marker="o", markersize=2.8,
            lw=0.7, label=gene_type_pretty[gt],
        )
        for gt in gene_type_order
    ],
    loc="center", ncol=3, frameon=False, fontsize=4.5,
    handletextpad=0.35, columnspacing=0.9,
)

gs_d_bins = gs_d_wrap[1].subgridspec(1, 5, wspace=0.55)
axes_d = [fig.add_subplot(gs_d_bins[0, j]) for j in range(5)]
for j, (xcol, ycol, ylabel, xlabel) in enumerate(prob_specs):
    ax = axes_d[j]
    for gt in gene_type_order:
        curve = prob_curves[(gt, xcol, ycol)]
        if curve.empty:
            continue
        xvals = curve["x"].to_numpy(dtype=float)
        if xcol == "max_median_counts":
            xvals = np.clip(xvals, 1e-1, None)
        color = gene_type_palette[gt]
        p_lo = curve["p_lo"].to_numpy(dtype=float)
        p_hi = curve["p_hi"].to_numpy(dtype=float)
        ax.fill_between(xvals, p_lo, p_hi, color=color, alpha=0.22, linewidth=0, zorder=1)
        ax.plot(xvals, p_lo, color=color, lw=0.7, alpha=0.7, zorder=2)
        ax.plot(xvals, p_hi, color=color, lw=0.7, alpha=0.7, zorder=2)
        ax.plot(
            xvals, curve["p"].to_numpy(dtype=float),
            "o-", color=color, lw=0.7, markersize=2.0, alpha=0.95, zorder=3,
        )
    if xcol == "max_median_counts":
        ax.set_xscale("log")
    else:
        ax.set_xlim(left=-0.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel, fontsize=label_fs)
    ax.set_ylabel(ylabel, fontsize=label_fs)
    ax.tick_params(axis="both", labelsize=tick_fs, pad=1, length=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Panel f — annotation ORs
gs_e_wrap = outer[4].subgridspec(2, 1, height_ratios=[0.18, 1.0], hspace=0.08)
ax_e_leg = fig.add_subplot(gs_e_wrap[0])
ax_e_leg.set_axis_off()
ax_or = fig.add_subplot(gs_e_wrap[1])

series_specs = [
    (annot_group, "eGene", "#7A7A7A", "o", "eGene · annotation", -0.30),
    (annot_group, "sGene", "#7A7A7A", "s", "sGene · annotation", -0.10),
    (joint_e_group, "eGene", "#2F2F2F", "o", "eGene · annotation + expr + N tissues", 0.10),
    (joint_s_group, "sGene", "#2F2F2F", "s", "sGene · annotation + expr + N tissues + N introns", 0.30),
]
x_base = {lab: i for i, lab in enumerate(panel_d_cat_order)}
plot_idx = or_df.set_index(["group", "modality", "label"])
or_lo_min, or_hi_max = 1.0, 1.0
legend_handles = []
for group, modality, color, marker, legend_lab, dx in series_specs:
    xs, ys, ylo, yhi, stars = [], [], [], [], []
    for lab in panel_d_cat_order:
        x = x_base[lab] + dx
        key = (group, modality, lab)
        if key not in plot_idx.index:
            ax_or.plot(x, 1.0, marker="x", color="0.55", ms=2.5, mew=0.7, zorder=3)
            continue
        row = plot_idx.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        or_hat, lo, hi, p = row["OR"], row["CI_low"], row["CI_high"], row["pvalue"]
        if not (np.isfinite(or_hat) and np.isfinite(lo) and np.isfinite(hi)):
            ax_or.plot(x, 1.0, marker="x", color="0.55", ms=2.5, mew=0.7, zorder=3)
            continue
        xs.append(x); ys.append(or_hat); ylo.append(lo); yhi.append(hi)
        if np.isfinite(p) and p < 1e-3:
            stars.append("***")
        elif np.isfinite(p) and p < 1e-2:
            stars.append("**")
        elif np.isfinite(p) and p < 0.05:
            stars.append("*")
        else:
            stars.append("")
        or_lo_min = min(or_lo_min, lo)
        or_hi_max = max(or_hi_max, hi)
    if not xs:
        continue
    ax_or.errorbar(
        xs, ys,
        yerr=[np.array(ys) - np.array(ylo), np.array(yhi) - np.array(ys)],
        fmt=marker, color=color, ecolor=color,
        markerfacecolor=color, markeredgecolor="k", markeredgewidth=0.25,
        elinewidth=0.65, markersize=2.8, label=legend_lab, zorder=3,
    )
    legend_handles.append(
        plt.Line2D(
            [0], [0], marker=marker, color=color, markerfacecolor=color,
            markeredgecolor="k", markeredgewidth=0.25, linestyle="None",
            markersize=3.2, label=legend_lab,
        )
    )
    for x, s, hi in zip(xs, stars, yhi):
        if s:
            ax_or.text(x, hi * 1.06, s, ha="center", va="bottom", color="black", fontsize=3.2)

ax_e_leg.legend(
    handles=legend_handles, loc="center", ncol=2, frameon=False,
    fontsize=3.6, handletextpad=0.3, columnspacing=0.8, labelspacing=0.25,
)

ax_or.axhline(1, color="0.5", ls="-", lw=0.45, zorder=1)
for _x in range(1, len(panel_d_cat_order)):
    ax_or.axvline(_x - 0.5, color="0.75", ls="-", lw=0.25, zorder=0)
ax_or.set_xticks(list(x_base.values()))
ax_or.set_xticklabels(
    list(x_base.keys()), rotation=0, ha="center", va="top",
    linespacing=0.85, fontsize=3.6,
)
ax_or.tick_params(axis="x", pad=1, length=0)
ax_or.tick_params(axis="y", labelsize=tick_fs, pad=1, length=2)
ax_or.set_ylabel("OR e/sGene", fontsize=label_fs)
ax_or.set_yscale("log", base=2)
y0 = max(or_lo_min, 1e-6) / 1.25
y1 = max(or_hi_max, 1.0) * 1.25 * 1.12
k0, k1 = int(np.floor(np.log2(y0))), int(np.ceil(np.log2(y1)))
ax_or.set_yticks([2 ** k for k in range(k0, k1 + 1)])
ax_or.set_yticklabels([rf"$2^{{{k}}}$" if k != 0 else "1" for k in range(k0, k1 + 1)])
ax_or.set_ylim(y0, y1)
ax_or.set_xlim(-0.6, len(panel_d_cat_order) - 0.4)
ax_or.spines["top"].set_visible(False)
ax_or.spines["right"].set_visible(False)
ax_or.spines["bottom"].set_visible(False)

_inv = fig.transFigure.inverted()
_x_left = min(
    _inv.transform(ax.transAxes.transform((-0.08, 0)))[0]
    for ax in (axes_a[0], axes_b[0], axes_c[0], axes_d[0], ax_or)
)
for ax, label in [
    (axes_a[0], "a"), (axes_b[0], "b"), (axes_c[0], "c"),
    (axes_d[0], "e"), (ax_or, "f"),
]:
    _y = _inv.transform(ax.transAxes.transform((0, 1.02)))[1]
    fig.text(_x_left, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
# Panel d sits in the right column of the c row
_xd = _inv.transform(ax_mod.transAxes.transform((-0.08, 0)))[0]
_yd = _inv.transform(ax_mod.transAxes.transform((0, 1.02)))[1]
fig.text(_xd, _yd, "d", fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_7_egene_sgene_odds.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_7_egene_sgene_odds.png", bbox_inches="tight")

## Print stats


In [ ]:
# Results: any-tissue eGene ORs among lncRNAs (one-at-a-time and joint models)
dgtex_or = plot_idx.loc[(annot_group, "eGene", "dev.\nexpr.")]
nhp_or = plot_idx.loc[(annot_group, "eGene", "primate\nexpr.")]
nhp_or_adj = plot_idx.loc[(joint_e_group, "eGene", "primate\nexpr.")]
print(f"dGTEx OR = {dgtex_or['OR']:.2f}; NHP OR = {nhp_or['OR']:.2f}")
print(f"adjusted NHP OR = {nhp_or_adj['OR']:.2f}, p = {nhp_or_adj['pvalue']:.2f}")

# Caption a,b: median of per-tissue medians (non-zero values, tissues with ≥5 genes)
for stat, plot_df, value_col, fmt in (
    ("read counts", expr_plot_df, "Expression", ".2g"),
    ("tested introns", junc_plot_df, "n_junctions", ".0f"),
):
    medians = {}
    for bt in gene_type_order:
        per_tissue = plot_df.loc[plot_df["biotype"] == bt].groupby("tissue")[value_col].agg(["median", "size"])
        medians[bt] = per_tissue.loc[per_tissue["size"] >= 5, "median"].median()
    print(f"median {stat}: " + "; ".join(f"{gene_type_pretty[bt]} = {m:{fmt}}" for bt, m in medians.items()))

# Caption c / Results: any-tissue eGene / sGene rate by biotype
for name, hollows in (("eGene", egene_hollows), ("sGene", sgene_hollows)):
    print(f"{name} any-tissue rate: " + "; ".join(f"{gene_type_pretty[gt]} = {pct:.0f}%" for gt, pct, _ in hollows))
